# Notebook 05 — Scene-Graph Evaluation

**Objective.** Compute object/attribute/relation/spatial PRF1 and tri-state
hallucination/evidence-support metrics for the structured (JSON) condition,
plus a descriptive abstention-phrase rate across all three conditions.

**Inputs.** The frozen raw predictions (`results/predictions/e1_*.jsonl` — 120
images x 2 models x 3 conditions = 720 records) + real GQA `val` scene graphs
on disk. No mocking, no toy ground truth beyond the tiny API demo below.

**Expected outputs.** Metric tables with bootstrap CIs, computed here for
inspection (this mirrors `scripts/evaluate_predictions.py`, which owns writing
the frozen `results/tables/e1_structured_scene_metrics.csv` — this notebook
cross-checks against that file rather than overwriting it, and saves its own
distinctly-named preview table instead).

_Companion notebook: runnable top-to-bottom, relative paths, no hidden state, saves artifacts to `results/`._

In [1]:
import sys, os
# Reusable logic lives in src/grounded_memory/. Notebooks call it.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from pathlib import Path
RESULTS = Path('..') / 'results'
from grounded_memory import __version__
print('grounded_memory', __version__)

grounded_memory 0.1.0


## Object metrics demo

Quick sanity-check of the deterministic metric functions on toy inputs before
using them on real predictions below.

In [2]:
from grounded_memory.evaluation.objects import object_prf1
from grounded_memory.evaluation.stats import bootstrap_ci
r = object_prf1(['dog','cat','car'], ['dog','cat','tree'])
print('F1', round(r.f1,3))
print('CI', bootstrap_ci([1,0,1,1,0,1], n_boot=2000))

F1 0.667
CI CI(point=0.6666666666666666, low=0.3333333333333333, high=1.0)


## Tri-state hallucination

Remember: an unannotated claim is NOT_VERIFIABLE, not automatically a
hallucination. `evaluation.grounding.verify_object_claim` /
`verify_relation_claim` implement that tri-state check against a real GQA
scene graph; used below via `evaluation.structured_scene.score_structured_scene`.

## Load the frozen E1 predictions

The real 720-prediction run (120 images x 2 models x {standard, grounded,
structured}), committed to the repo. Loaded read-only — nothing here
regenerates or overwrites these files (see `results/predictions/PROVENANCE_NOTE.md`).

In [3]:
import json
from collections import defaultdict

def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line]

predictions_dir = RESULTS / 'predictions'
files = sorted(predictions_dir.glob('e1_*.jsonl'))  # top-level only -- skips archive/ and logs/
print(f'{len(files)} frozen prediction files:')
for f in files:
    print(' ', f.name)

records_by_group: dict[tuple[str, str], list[dict]] = defaultdict(list)
for f in files:
    for rec in load_jsonl(f):
        condition = rec.get('metrics', {}).get('condition', '?')
        key = (rec.get('model_id', '?'), condition)
        records_by_group[key].append(rec)

total = sum(len(v) for v in records_by_group.values())
print(f'\n{total} total predictions loaded across {len(records_by_group)} (model, condition) groups.')

6 frozen prediction files:
  e1_llava_7b_grounded__e1_20260821T000539Z.jsonl
  e1_llava_7b_standard__e1_20260820T080715Z.jsonl
  e1_llava_7b_structured__e1_20260821T004038Z.jsonl
  e1_qwen3-vl_8b_grounded__e1_20260820T154255Z.jsonl
  e1_qwen3-vl_8b_standard__e1_20260819T211021Z.jsonl
  e1_qwen3-vl_8b_structured__e1_20260820T050820Z.jsonl

720 total predictions loaded across 6 (model, condition) groups.


## Parser-failure accounting

Only the `structured` condition asks for JSON; `standard`/`grounded` are free
text and never attempt parsing (`parse_ok` is only meaningful for structured).

In [4]:
import pandas as pd

parse_rows = []
for (model, condition), records in sorted(records_by_group.items()):
    n = len(records)
    if condition == 'structured':
        parse_ok = sum(1 for r in records if r.get('parse_ok'))
        parse_fail = n - parse_ok
    else:
        parse_ok = parse_fail = None
    parse_rows.append({'model_id': model, 'condition': condition, 'n': n,
                        'parse_ok': parse_ok, 'parse_fail': parse_fail})

parse_df = pd.DataFrame(parse_rows)
parse_df

,model_id,condition,n,parse_ok,parse_fail
0,ollama/llava:7b,grounded,120,NaN,NaN
1,ollama/llava:7b,standard,120,NaN,NaN
2,ollama/llava:7b,structured,120,111.0,9.0
3,ollama/qwen3-vl:8b,grounded,120,NaN,NaN
4,ollama/qwen3-vl:8b,standard,120,NaN,NaN
5,ollama/qwen3-vl:8b,structured,120,115.0,5.0


## Load real GQA scene graphs

`gqa.root` in `configs/datasets.yaml` is repo-root-relative -- resolve it
against the repo root explicitly, since the notebook kernel's cwd is
`notebooks/`, not the repo root (same fix as notebooks 02/03/04).

In [5]:
from grounded_memory.config import load_config
from grounded_memory.data.gqa import load_scene_graphs

data_cfg = load_config('datasets')
REPO_ROOT = Path('..').resolve()
gqa_root = Path(data_cfg['gqa']['root'])
if not gqa_root.is_absolute():
    gqa_root = (REPO_ROOT / gqa_root).resolve()
split = data_cfg['gqa']['benchmark_split']
scene_graph_file = gqa_root / data_cfg['gqa']['scene_graphs'][split]
print(f'Scene graphs: {scene_graph_file} (exists: {scene_graph_file.exists()})')

scene_graphs = load_scene_graphs(scene_graph_file)
print(f'{len(scene_graphs)} scene graphs loaded ({split} split).')

Scene graphs: /Users/sanjad/ai/projects/grounded-multimodal-memory/data/raw/gqa/sceneGraphs/val_sceneGraphs.json (exists: True)


10696 scene graphs loaded (val split).

## Score the structured condition against ground truth

Only the structured (JSON) condition can be scored against the scene graph
this way -- `standard`/`grounded` predictions are free text with no
extraction path yet (see `evaluation/structured_scene.py`'s module docstring).
`relationship`/`spatial` exclude GQA's exhaustive left/right positional
annotations (95.7% of all relation instances in this manifest); those are
scored separately as `positional` — see the same docstring for why.

In [6]:
from grounded_memory.evaluation.structured_scene import score_structured_scene
from grounded_memory.scene.schema import SceneRepresentation

METRIC_NAMES = [
    'object_precision', 'object_recall', 'object_f1',
    'attribute_precision', 'attribute_recall', 'attribute_f1',
    'relationship_precision', 'relationship_recall', 'relationship_f1',
    'spatial_precision', 'spatial_recall', 'spatial_f1',
    'positional_precision', 'positional_recall', 'positional_f1',
    'hallucination_rate', 'evidence_support_rate',
]

scored: dict[tuple[str, str], dict[str, list[float]]] = defaultdict(lambda: defaultdict(list))
missing_scene_graph = 0
for (model, condition), records in records_by_group.items():
    if condition != 'structured':
        continue
    for rec in records:
        if not rec.get('parse_ok') or not rec.get('parsed_prediction'):
            continue
        sg = scene_graphs.get(rec['sample_id'])
        if sg is None:
            missing_scene_graph += 1
            continue
        scene = SceneRepresentation.model_validate(rec['parsed_prediction'])
        score = score_structured_scene(scene, sg)
        bucket = scored[(model, condition)]
        for name, prf1 in (
            ('object', score.objects), ('attribute', score.attributes),
            ('relationship', score.relationships), ('spatial', score.spatial),
            ('positional', score.positional),
        ):
            bucket[f'{name}_precision'].append(prf1.precision)
            bucket[f'{name}_recall'].append(prf1.recall)
            bucket[f'{name}_f1'].append(prf1.f1)
        bucket['hallucination_rate'].append(score.hallucination_rate)
        bucket['evidence_support_rate'].append(score.evidence_support_rate)

n_scored = sum(len(next(iter(v.values()))) for v in scored.values())
print(f'Scored {n_scored} structured predictions across {len(scored)} model group(s); '
      f'{missing_scene_graph} skipped (no matching scene graph).')

Scored 226 structured predictions across 2 model group(s); 0 skipped (no matching scene graph).


## Metric table with bootstrap CIs

Same bootstrap settings as `configs/evaluation.yaml` (`statistics:`), so this
matches the frozen table exactly if nothing about the metric code has changed.

In [7]:
eval_cfg = load_config('evaluation')
stats_cfg = eval_cfg.get('statistics', {})
n_boot = stats_cfg.get('bootstrap_samples', 10000)
ci = stats_cfg.get('ci', 0.95)
seed = stats_cfg.get('seed', 20260815)

rows = []
for (model, condition), metrics in sorted(scored.items()):
    n = len(next(iter(metrics.values())))
    for name in METRIC_NAMES:
        result = bootstrap_ci(metrics[name], n_boot=n_boot, ci=ci, seed=seed)
        rows.append({'model_id': model, 'condition': condition, 'metric': name, 'n': n,
                     'mean': result.point, 'ci_low': result.low, 'ci_high': result.high})

metrics_df = pd.DataFrame(rows)
metrics_df

,model_id,condition,metric,n,mean,ci_low,ci_high
0,ollama/llava:7b,structured,object_precision,111,0.537538,0.467718,0.606456
1,ollama/llava:7b,structured,object_recall,111,0.088759,0.074333,0.103910
2,ollama/llava:7b,structured,object_f1,111,0.146400,0.124647,0.168793
3,ollama/llava:7b,structured,attribute_precision,111,0.064136,0.029601,0.106178
4,ollama/llava:7b,structured,attribute_recall,111,0.012739,0.005365,0.022044
5,ollama/llava:7b,structured,attribute_f1,111,0.019733,0.008746,0.033016
6,ollama/llava:7b,structured,relationship_precision,111,0.040541,0.009009,0.081081
7,ollama/llava:7b,structured,relationship_recall,111,0.007347,0.001287,0.015337
8,ollama/llava:7b,structured,relationship_f1,111,0.011311,0.002252,0.022525
9,ollama/llava:7b,structured,spatial_precision,111,0.022523,0.000000,0.054054


## Cross-check against the frozen table

`scripts/evaluate_predictions.py` already computed and froze
`results/tables/e1_structured_scene_metrics.csv` from this same data. If this
notebook's numbers agree, that's a real reproducibility check, not a rerun of
the same code path with nothing to verify against.

In [8]:
frozen = pd.read_csv(RESULTS / 'tables' / 'e1_structured_scene_metrics.csv')
merged = metrics_df.merge(frozen, on=['model_id', 'condition', 'metric'], suffixes=('_nb', '_frozen'))
merged['abs_diff'] = (merged['mean_nb'] - merged['mean_frozen']).abs()
print(f'{len(merged)}/{len(metrics_df)} rows matched the frozen table.')
print('max |mean_nb - mean_frozen| =', merged['abs_diff'].max())
merged[['model_id', 'condition', 'metric', 'mean_nb', 'mean_frozen', 'abs_diff']].sort_values('abs_diff', ascending=False).head()

34/34 rows matched the frozen table.
max |mean_nb - mean_frozen| = 1.1102230246251565e-16


,model_id,condition,metric,mean_nb,mean_frozen,abs_diff
33,ollama/qwen3-vl:8b,structured,evidence_support_rate,0.947826,0.947826,1.110223e-16
10,ollama/llava:7b,structured,spatial_recall,0.002789,0.002789,8.803722e-17
31,ollama/qwen3-vl:8b,structured,positional_f1,0.012854,0.012854,8.153200e-17
4,ollama/llava:7b,structured,attribute_recall,0.012739,0.012739,7.112366e-17
1,ollama/llava:7b,structured,object_recall,0.088759,0.088759,6.938894e-17


## Save a notebook-specific preview table

Distinct filename -- never collides with, or overwrites, the frozen table
that `scripts/evaluate_predictions.py` owns.

In [9]:
preview_path = RESULTS / 'tables' / 'nb05_structured_scene_metrics_preview.csv'
metrics_df.to_csv(preview_path, index=False)
print(f'wrote {len(metrics_df)} rows -> {preview_path}')

wrote 34 rows -> ../results/tables/nb05_structured_scene_metrics_preview.csv


## Hallucination rate is structurally always 0.0 right now

Worth seeing directly, not just asserted: `verify_object_claim`/
`verify_relation_claim` (`evaluation/grounding.py`) only ever emit
`SUPPORTED` or `NOT_VERIFIABLE` today -- there is no code path that emits
`CONTRADICTED` (that would require detecting, e.g., "model says red, scene
graph says blue", which isn't implemented). So `hallucination_rate` reads
0.0 for every group above by construction, not because either model never
hallucinates. `evidence_support_rate` (supported / (supported+contradicted),
which today is just supported / supported) is the metric that actually
carries signal here.

In [10]:
from collections import Counter

from grounded_memory.evaluation.grounding import verify_object_claim

label_counts: dict[str, Counter] = {}
for (model, condition), records in records_by_group.items():
    if condition != 'structured':
        continue
    counts = Counter()
    for rec in records:
        if not rec.get('parse_ok') or not rec.get('parsed_prediction'):
            continue
        sg = scene_graphs.get(rec['sample_id'])
        if sg is None:
            continue
        for entity in rec['parsed_prediction'].get('entities', []):
            counts[verify_object_claim(entity.get('label', ''), sg.raw)] += 1
    label_counts[model] = counts

for model, counts in label_counts.items():
    total = sum(counts.values())
    print(f'{model}: {dict(counts)}  (total object claims={total})')
    print('  -> no CONTRADICTED possible today; NOT_VERIFIABLE only means "not annotated", not "wrong"')

ollama/llava:7b: {<EvidenceLabel.NOT_VERIFIABLE: 'not_verifiable'>: 187, <EvidenceLabel.SUPPORTED: 'supported'>: 238}  (total object claims=425)
  -> no CONTRADICTED possible today; NOT_VERIFIABLE only means "not annotated", not "wrong"
ollama/qwen3-vl:8b: {<EvidenceLabel.SUPPORTED: 'supported'>: 645, <EvidenceLabel.NOT_VERIFIABLE: 'not_verifiable'>: 283}  (total object claims=928)
  -> no CONTRADICTED possible today; NOT_VERIFIABLE only means "not annotated", not "wrong"


## Descriptive abstention-phrase rate

E1's three prompt conditions never ask a question that *requires* abstention
-- there's no `requires_abstention` ground truth for open-ended scene
description / structured-JSON prompts (that concept exists in
`scene/grounding.py`'s `generate_probes()`, for QA-style probes, but it isn't
wired into any experiment run yet). So `evaluation.abstention.abstention_metrics()`
(which needs `requires_abstention` labels per item) doesn't apply to this
data and isn't used here.

What *is* real: whether a model's raw output happened to contain one of the
abstention phrases from `configs/evaluation.yaml` (`is_abstention`), of its
own accord, across all three conditions. Read this as "how often did the
model hedge/decline unprompted" -- not as an accuracy number.

In [11]:
from grounded_memory.evaluation.abstention import is_abstention

abstain_rows = []
for (model, condition), records in sorted(records_by_group.items()):
    n = len(records)
    n_abstain = sum(1 for r in records if is_abstention(r.get('prediction', '')))
    abstain_rows.append({
        'model_id': model, 'condition': condition, 'n': n,
        'raw_abstention_phrase_rate': n_abstain / n if n else 0.0,
    })

abstain_df = pd.DataFrame(abstain_rows)
abstain_df

,model_id,condition,n,raw_abstention_phrase_rate
0,ollama/llava:7b,grounded,120,0.075000
1,ollama/llava:7b,standard,120,0.016667
2,ollama/llava:7b,structured,120,0.291667
3,ollama/qwen3-vl:8b,grounded,120,0.941667
4,ollama/qwen3-vl:8b,standard,120,0.000000
5,ollama/qwen3-vl:8b,structured,120,0.266667


## Findings

1. This notebook's numbers reproduce the frozen
   `results/tables/e1_structured_scene_metrics.csv` exactly (34/34 rows
   matched, max |mean_nb - mean_frozen| = 1.1e-16, i.e. float noise) — the
   metric code is deterministic and this read-only re-derivation agrees with
   the script that froze the table.
2. Object recall is low for both models on the structured condition
   (qwen3-vl:8b 0.181 [0.162, 0.200], llava:7b 0.089 [0.074, 0.104]) —
   neither model enumerates anywhere close to GQA's full annotated object
   set (these images have 50-75+ annotated objects each; see
   `data/manifests/gqa_benchmark_v1.csv`) — while object precision is much
   higher (qwen3-vl:8b 0.588, llava:7b 0.538): what both models *do* name
   tends to be correct, they just don't name much of the scene.
3. `evidence_support_rate` (the metric that actually carries
   hallucination-adjacent signal, see the section above) is high for both —
   qwen3-vl:8b 0.948 [0.904, 0.983], llava:7b 0.757 [0.676, 0.829] — but
   `hallucination_rate` is 0.0 for both by construction (the tri-state
   verifier never emits CONTRADICTED yet), so "0% hallucination" must not be
   read as "the models never hallucinate."
4. `positional` (GQA's exhaustive left/right pairwise annotations, scored
   separately from other relations) shows a real qualitative difference:
   llava:7b scores exactly 0.0 precision/recall across all 111 of its
   structured predictions (never produces a matching left/right claim) vs
   qwen3-vl:8b's 0.112 precision [0.086, 0.139] — some real signal, though
   still very low recall (0.007) since GQA's positional annotation is far
   denser than any model's answer.
5. attribute/relationship/spatial F1 are all in the 0.003-0.06 range for
   both models — the weakest categories by a wide margin, well below object
   F1 — consistent with `notes/done_so_far.md` (Phase 12)'s note that this
   is a believable-but-low signal, not a methodology artifact (that
   artifact, left/right dominating the raw relation reference set, was
   already isolated into `positional` above).
6. Structured-condition JSON parsing: llava:7b failed to parse on 9/120
   images (7.5%), qwen3-vl:8b on 5/120 (4.2%) — both models mostly produce
   valid structured JSON, but not universally.
7. The raw abstention-phrase rate has a striking outlier: qwen3-vl:8b's
   *grounded* condition hits 0.942 (113/120 responses contain a phrase like
   "not visible" or "cannot be determined"), far above its own `standard`
   condition (0.0) or `structured` condition (0.267), and far above
   llava:7b's `grounded` rate (0.075). This is a real, surprising number
   worth investigating further (e.g. whether the grounded prompt template
   itself invites per-claim hedging language from qwen3-vl specifically) —
   not explained here, since that would require inspecting raw grounded
   predictions rather than guessing.

> Do not populate tables with invented numbers. If the full model/data is unavailable, leave clearly-labeled placeholders.